In [1]:
import cv2 
import mediapipe as mp
import time

# mediapipe tasks + GESTURE RECOGNISER setup 
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

GestureRecognizer = mp.tasks.vision.GestureRecognizer
GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# hand skeleton connections
hand_connections = [
    (0,1), (1,2), (2,3), (3,4),       # thumb
    (0,5), (5,6), (6,7), (7,8),       # index finger
    (0,9), (9,10), (10,11), (11,12),  # middle finger
    (0,13),(13,14), (14,15), (15,16), # ring finger
    (0,17), (17,18), (18,19),(19,20),  # pinky
    (5, 9), (9, 13), (13, 17)
    ]

def draw_hand_landmarks(image, hand_landmarks): 
    # draw hand skeleton(s) with lines and dots
    height, width, _ = image.shape
    
    # draw connections 
    for connection in hand_connections:
        start_index, end_index = connection
        
        # get start point
        start = hand_landmarks[start_index]
        start_x = int(start.x * width)
        start_y = int(start.y * height)
        
        # get end point
        end = hand_landmarks[end_index]
        end_x = int(end.x * width)
        end_y = int(end.y * height)
        
        # draw line
        cv2.line(image, (start_x, start_y), (end_x, end_y), (0, 255, 0), 2)
        
        
    # draw landmarks 
    for landmark in hand_landmarks:
        x = int(landmark.x * width)
        y = int(landmark.y * height)
        cv2.circle(image, (x, y), 5, (0, 0, 255), -1)

In [4]:
# gesture recogniser 
options = GestureRecognizerOptions(
    base_options=BaseOptions(model_asset_path='gesture_recognizer.task'),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2)
    

cap = cv2.VideoCapture(0) # to live stream from my laptop
timestamp = 0

with GestureRecognizer.create_from_options(options) as recognizer:
    while cap.isOpened():
        success, image = cap.read()
        if not success:
            break
        
        image = cv2.flip(image, 1)
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
        
        timestamp += 33
        results = recognizer.recognize_for_video(mp_image, timestamp)
        
        # draw hand skeletons
        if results.hand_landmarks:
            for hand in results.hand_landmarks:
                draw_hand_landmarks(image, hand)
                
        
        # Show what's detected
        if results.gestures:
            gesture = results.gestures[0][0]  # First hand, top prediction
            name = gesture.category_name
            confidence = gesture.score
            
            # Big text display
            cv2.putText(image, name, 
                        (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 4)
            cv2.putText(image, f"{confidence:.0%}", 
                        (50, 160), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 3)
        else:
            cv2.putText(image, "No gesture",
                        (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 4)
        
        cv2.imshow('Practice Phase', image)
        
        if cv2.waitKey(5) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1777562935.060222 16666070 hand_gesture_recognizer_graph.cc:257] Custom gesture classifier is not defined.
I0000 00:00:1777562935.088331 16666070 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M1
W0000 00:00:1777562935.100806 16666075 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777562935.227022 16666075 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777562935.228474 16666079 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777562935.228554 16666079 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777562935.253047 166660

: 